# DiffuLLaMA-7B: the four remaining experiments

Runs `time_curve`, `attention_entropy`, `logit_lens`, and `pos_probe` for DiffuLLaMA-7B so it matches Dream-7B's coverage.

**Runtime: A100.** A T4 has 16GB and cannot hold the model in bf16 alongside the eager attention tensors. Set it under *Runtime -> Change runtime type* before running anything.

Two things that have silently produced garbage before, both now guarded in code but worth knowing:

1. **`transformers` must be exactly 4.44.2.** DiffuLLaMA's `attention_patch.py` replaces `LlamaModel.forward` with the 4.44-era implementation and patches `LlamaFlashAttention2`, which later releases deleted. Cell 2 pins it and **the runtime must be restarted afterwards**.
2. **Attention must be eager.** `sdpa` and `flash_attention_2` accept `output_attentions=True` and return nothing, which turns the entire analysis into zeros. `configs/models/diffullama_7b.yaml` sets this.

The head search is already done and committed, so this notebook does not repeat it.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import subprocess

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
).stdout.strip()

mib = int(gpu.rsplit(",", 1)[-1].strip().split()[0]) if gpu else 0
if mib < 30000:
    raise SystemExit(
        f"{gpu!r} has {mib} MiB. DiffuLLaMA-7B in bf16 needs ~14GB for weights plus "
        "~0.5GB per sentence of eager attention. Switch to an A100."
    )
print("OK:", gpu)

## 2. Install

**This cell restarts the runtime at the end.** That is expected — Colab preloads a newer `transformers`, and downgrading it in-process leaves the old module cached. Re-run from cell 3 afterwards.

In [ ]:
REPO = "github.com/Dabsoysauce/latentrelationsondlm.git"
# The fixes that unblock these four experiments live here until it is merged.
BRANCH = "diffullama-experiments"
SRC = "/content/dlmresearch/src"

import os

if not os.path.exists("/content/dlmresearch"):
    # Private repo. Preferred: Colab left sidebar -> key icon -> add a secret
    # named GITHUB_TOKEN (a PAT with read access to this repo), then enable it
    # for this notebook. Falls back to a hidden prompt.
    token = None
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        print(f"no Colab secret ({type(exc).__name__}); prompting instead")

    if not token:
        import getpass

        token = getpass.getpass("GitHub token (input hidden): ")

    # The token stays in this variable only; it is not echoed or written to disk.
    !git clone --quiet --branch {BRANCH} https://{token}@{REPO} /content/dlmresearch
    del token

%cd /content/dlmresearch
!git log --oneline -1
!pip install -q -e ".[diffullama,probe]"

# An editable install registers the package through a .pth file, and .pth files
# are only processed at interpreter startup -- so `import dlmrel` raises
# ModuleNotFoundError in the very session that installed it. Point at the source
# root directly rather than making the whole notebook depend on a restart.
import importlib, sys

if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

import dlmrel

print("dlmrel:", dlmrel.__file__)

import transformers

print("transformers:", transformers.__version__)
if transformers.__version__ != "4.44.2":
    print("Restarting so the pin takes effect. Re-run from cell 3 (not cell 2).")
    os.kill(os.getpid(), 9)

## 3. Verify the environment

In [ ]:
%cd /content/dlmresearch

# Re-applied because this cell is the entry point after a runtime restart, and
# a restart clears the sys.path entry added during install.
import importlib, sys

SRC = "/content/dlmresearch/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

import dlmrel, numpy, torch, transformers

print("dlmrel      ", dlmrel.__file__)
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("numpy       ", numpy.__version__)

assert transformers.__version__ == "4.44.2", (
    f"transformers is {transformers.__version__}; DiffuLLaMA's attention patch "
    "needs 4.44.2. Re-run cell 2 and let it restart the runtime."
)

# The patch targets a class later releases removed. Fail here rather than
# midway through a multi-hour run.
from transformers.models.llama import modeling_llama

assert hasattr(modeling_llama, "LlamaFlashAttention2"), (
    "LlamaFlashAttention2 is missing, so attention_patch.py will fail on import."
)

assert torch.cuda.is_available(), "no GPU visible to torch"
print("\nenvironment OK")

## 4. Data

The splits are committed, so this only regenerates them if they are missing. They are carved from a pool that all three models' tokenizers accept, which is what makes the test split byte-identical across models — without it a ~1% pool difference dropped split overlap to 73%.

In [ ]:
from pathlib import Path

OUT = Path("results/diffullama_7b")

# offset_null.csv is the only prepare-data output the experiments read directly;
# the Example objects themselves are rebuilt from the treebank either way.
if (OUT / "offset_null.csv").exists():
    print("offset null table already present")
else:
    !dlmrel prepare-data --model diffullama_7b

import pandas as pd

mine = pd.read_csv(OUT / "sentences_test.csv")["sentence"]
ref = pd.read_csv("results/dream_7b/sentences_test.csv")["sentence"]
assert list(mine) == list(ref), "test split diverged from Dream-7B"
print(f"test split matches Dream-7B exactly ({len(mine)} sentences)")

print("\nOffset-null baseline (what every head is measured against):")
print(pd.read_csv(OUT / "offset_null.csv").to_string(index=False))

## 5. Load the model once

Loading is the expensive part, so all four experiments share one loaded model rather than going through `dlmrel run` four times.

In [ ]:
import importlib, json

from dlmrel.cli import EXPERIMENTS, _build_config, _experiment_config, _model_config

MODEL = "diffullama_7b"
model_cfg = _model_config(MODEL)
print(model_cfg)

adapter_mod = importlib.import_module(f"dlmrel.models.{model_cfg['adapter']}")
model, tokenizer, meta = adapter_mod.load(model_cfg)

Path(f"results/{MODEL}/model_meta.json").write_text(json.dumps(meta, indent=2))
print(meta)

### Sanity check the loaded weights

The checkpoint is stored under a `denoise_model.*` wrapper namespace. Loading it the obvious way leaves the backbone randomly initialised with only a warning — that is what produced the first 7B head search at chance (0.078, 0/144 heads above null). This reproduces the known prev-token head instead of trusting the load.

In [ ]:
import torch

probe = torch.tensor(
    [[tokenizer.bos_token_id] + tokenizer.encode("The cat sat on the mat.", add_special_tokens=False)],
    device=model.device,
)
_, attentions = model.forward_attentions(probe)
assert attentions and attentions[0] is not None, "no attention weights: not eager"

n_layers, n_heads = len(attentions), attentions[0].shape[1]
print(f"{n_layers} layers x {n_heads} heads")

# A random model has no head that consistently attends one position back.
best = 0.0
for layer in range(n_layers):
    a = attentions[layer][0].float()
    for h in range(n_heads):
        m = a[h, 1:, :].clone()
        m[:, 0] = 0.0  # attention sink
        pred = m.argmax(dim=-1).cpu()
        best = max(best, (pred == torch.arange(pred.shape[0])).float().mean().item())

print(f"best prev-token score: {best:.3f}")
assert best > 0.5, (
    f"no prev-token head found ({best:.3f}); the backbone is probably randomly "
    "initialised. Do not run the experiments on this model."
)
print("weights look real")

## 6. Run the four experiments

Ordered cheapest first, so a failure surfaces before hours are spent. Each writes to `results/diffullama_7b/<experiment>/` and is skipped if its output already exists, which makes the notebook resumable after a disconnect.

In [ ]:
import inspect, time, traceback

# (experiment, file that means "already done")
PLAN = [
    ("attention_entropy", "attention_entropy.csv"),
    ("logit_lens", "logit_lens.csv"),
    ("pos_probe", "pos_probe.csv"),
    ("time_curve", "curve_aggregate.csv"),
]

results = {}
for name, marker in PLAN:
    out = Path(f"results/{MODEL}/{name}")
    if (out / marker).exists():
        print(f"== {name}: already done, skipping")
        results[name] = "skipped"
        continue

    cfg = _build_config(MODEL, model_cfg, _experiment_config(name))
    experiment = importlib.import_module(EXPERIMENTS[name])
    extra = {"meta": meta} if "meta" in inspect.signature(experiment.run).parameters else {}

    print(f"\n{'=' * 60}\n== {name}\n{'=' * 60}", flush=True)
    start = time.time()
    try:
        experiment.run(model, tokenizer, cfg, out, **extra)
        results[name] = f"ok ({time.time() - start:.0f}s)"
    except Exception:
        traceback.print_exc()
        results[name] = "FAILED"
    print(f"== {name}: {results[name]}", flush=True)

print("\nSummary")
for k, v in results.items():
    print(f"  {k:20s} {v}")

## 7. Check the outputs before committing

In [ ]:
for name, _ in PLAN:
    d = Path(f"results/{MODEL}/{name}")
    files = sorted(p.name for p in d.glob("*.csv")) if d.exists() else []
    print(f"{name:20s} {files or 'MISSING'}")

curve = Path(f"results/{MODEL}/time_curve/masked_state_vs_null.csv")
if curve.exists():
    print("\nMasked-state accuracy vs the offset null (the H2 result):")
    print(pd.read_csv(curve).to_string(index=False))

## 8. Download the results

`curve_raw.csv` is large and is the only file that can be regenerated from the others, so it is excluded. Unzip into the repo root and commit on a branch.

In [ ]:
!cd /content/dlmresearch && zip -qr /content/diffullama_7b_results.zip results/diffullama_7b -x "*/curve_raw.csv"

from google.colab import files

files.download("/content/diffullama_7b_results.zip")